# 04 — Community Detection


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Community Detection

Louvain and Leiden on undirected graph using direct edges only (is_direct=True).


In [ ]:
import igraph as ig
import leidenalg
import numpy as np
from networkx.algorithms.community import louvain_communities, modularity

# ── Build undirected graph using direct edges only ──────────────────────────
direct_edges = edge_df[edge_df['is_direct']]

G_und = nx.Graph()
G_und.add_nodes_from(G.nodes())
G_und.add_edges_from(zip(direct_edges['source_target'], direct_edges['dest_target']))

print(f"Undirected graph: {G_und.number_of_nodes()} nodes, {G_und.number_of_edges()} edges")
print(f"  (Original directed G: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")
print(f"  Direct edges used: {len(direct_edges)}")
print(f"  Isolated nodes (no direct edges): {sum(1 for n in G_und.nodes() if G_und.degree(n) == 0)}")

# ── Louvain (NetworkX) ──────────────────────────────────────────────────────
louvain_comms = louvain_communities(G_und, resolution=1.0, seed=42)
louvain_mod = modularity(G_und, louvain_comms)

louvain_node_comm = {}
for i, comm in enumerate(louvain_comms):
    for n in comm:
        louvain_node_comm[n] = i

print(f"\nLouvain:  {len(louvain_comms)} communities, modularity = {louvain_mod:.4f}")
for i, comm in enumerate(louvain_comms):
    print(f"  C{i:02d}: {len(comm)} targets — {sorted(comm)}")

# ── Leiden (igraph + leidenalg) ─────────────────────────────────────────────
ig_g = ig.Graph.from_networkx(G_und)
_node_names = ig_g.vs['_nx_name']

leiden_part = leidenalg.find_partition(ig_g, leidenalg.ModularityVertexPartition, seed=42)
leiden_node_comm = {_node_names[i]: leiden_part.membership[i] for i in range(ig_g.vcount())}
leiden_comms = [
    {n for n, c in leiden_node_comm.items() if c == cid}
    for cid in range(max(leiden_part.membership) + 1)
]
leiden_mod = modularity(G_und, leiden_comms)

print(f"\nLeiden:   {len(leiden_comms)} communities, modularity = {leiden_mod:.4f}")
for i, comm in enumerate(leiden_comms):
    print(f"  C{i:02d}: {len(comm)} targets — {sorted(comm)}")

# ── Choose working partition for downstream cells ───────────────────────────
if leiden_mod >= louvain_mod:
    node_comm = leiden_node_comm
    communities = leiden_comms
    print(f"\nUsing Leiden partition (modularity {leiden_mod:.4f} >= Louvain {louvain_mod:.4f})")
else:
    node_comm = louvain_node_comm
    communities = louvain_comms
    print(f"\nUsing Louvain partition (modularity {louvain_mod:.4f} > Leiden {leiden_mod:.4f})")

n_communities = len(communities)

## Resolution Parameter Tuning

Sweep resolution parameter, evaluate modularity.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

resolutions = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
results = []

for r in resolutions:
    # Louvain
    lv_comms = louvain_communities(G_und, resolution=r, seed=42)
    lv_mod = modularity(G_und, lv_comms)

    # Leiden (RBConfigurationVertexPartition supports resolution_parameter)
    ld_part = leidenalg.find_partition(
        ig_g, leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=r, seed=42,
    )
    ld_comms = [
        {_node_names[j] for j, c in enumerate(ld_part.membership) if c == cid}
        for cid in range(max(ld_part.membership) + 1)
    ]
    ld_mod = modularity(G_und, ld_comms)

    results.append({
        'resolution': r,
        'louvain_n_comms': len(lv_comms),
        'louvain_modularity': lv_mod,
        'leiden_n_comms': len(ld_comms),
        'leiden_modularity': ld_mod,
    })
    print(f"r={r:.2f}  Louvain: {len(lv_comms):3d} comms Q={lv_mod:.4f}  "
          f"Leiden: {len(ld_comms):3d} comms Q={ld_mod:.4f}")

sweep_df = pd.DataFrame(results)
display(sweep_df)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sweep_df['resolution'], sweep_df['louvain_modularity'],
         marker='o', color='steelblue', label='Louvain')
ax1.plot(sweep_df['resolution'], sweep_df['leiden_modularity'],
         marker='s', color='darkorange', label='Leiden')
ax1.axvline(1.0, color='red', linestyle='--', alpha=0.6, label='r=1.0 (default)')
ax1.set_xlabel('Resolution')
ax1.set_ylabel('Modularity')
ax1.set_title('Modularity vs Resolution')
ax1.legend()

ax2.plot(sweep_df['resolution'], sweep_df['louvain_n_comms'],
         marker='o', color='steelblue', label='Louvain')
ax2.plot(sweep_df['resolution'], sweep_df['leiden_n_comms'],
         marker='s', color='darkorange', label='Leiden')
ax2.axvline(1.0, color='red', linestyle='--', alpha=0.6, label='r=1.0 (default)')
ax2.set_xlabel('Resolution')
ax2.set_ylabel('Number of Communities')
ax2.set_title('Community Count vs Resolution')
ax2.legend()

plt.tight_layout()
plt.show()

# ── Identify best resolution ────────────────────────────────────────────────
best_louvain = sweep_df.loc[sweep_df['louvain_modularity'].idxmax()]
best_leiden = sweep_df.loc[sweep_df['leiden_modularity'].idxmax()]
print(f"\nBest Louvain: r={best_louvain['resolution']:.2f}, "
      f"Q={best_louvain['louvain_modularity']:.4f}, "
      f"{int(best_louvain['louvain_n_comms'])} communities")
print(f"Best Leiden:  r={best_leiden['resolution']:.2f}, "
      f"Q={best_leiden['leiden_modularity']:.4f}, "
      f"{int(best_leiden['leiden_n_comms'])} communities")

## Community Characterisation

Per community: target count, compile time, SLOC, cohesion/coupling, codegen count.


In [ ]:
target_metrics = target_df.set_index('cmake_target')
codegen_targets_set = set(target_df.loc[target_df['codegen_file_count'].fillna(0) > 0, 'cmake_target'])

rows = []
for cid, comm in enumerate(communities):
    nodes = list(comm)
    subg = G_und.subgraph(comm)
    internal_edges = subg.number_of_edges()
    external_edges = sum(1 for u in comm for v in G_und.neighbors(u) if v not in comm)
    total_edges = internal_edges + external_edges
    cohesion = internal_edges / total_edges if total_edges > 0 else 0.0
    coupling = external_edges / total_edges if total_edges > 0 else 0.0

    def safe_col_sum(col):
        vals = [target_metrics.loc[n, col] for n in nodes
                if n in target_metrics.index and pd.notna(target_metrics.loc[n, col])]
        return int(sum(vals)) if vals else 0

    codegen_count = sum(1 for n in nodes if n in codegen_targets_set)

    rows.append({
        'community_id': cid,
        'target_count': len(nodes),
        'compile_time_sum_ms': safe_col_sum('compile_time_sum_ms'),
        'code_lines_total': safe_col_sum('code_lines_total'),
        'internal_edges': internal_edges,
        'external_edges': external_edges,
        'cohesion_ratio': round(cohesion, 4),
        'coupling_ratio': round(coupling, 4),
        'codegen_target_count': codegen_count,
        'codegen_compile_time_ms': safe_col_sum('codegen_compile_time_sum_ms'),
    })

comm_df = pd.DataFrame(rows).sort_values('compile_time_sum_ms', ascending=False).reset_index(drop=True)
comm_df['compile_time_pct'] = (comm_df['compile_time_sum_ms'] / comm_df['compile_time_sum_ms'].sum() * 100).round(1)

display(comm_df)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Target count per community
ax = axes[0, 0]
sns.barplot(data=comm_df, x='community_id', y='target_count', color='steelblue', ax=ax)
ax.set_xlabel('Community')
ax.set_ylabel('Target Count')
ax.set_title('Targets per Community')

# 2. Compile time per community (with codegen overlay)
ax = axes[0, 1]
x = comm_df['community_id'].astype(str)
ax.bar(x, comm_df['compile_time_sum_ms'], color='steelblue', label='Total compile time')
ax.bar(x, comm_df['codegen_compile_time_ms'], color='darkorange', label='Codegen compile time', alpha=0.9)
ax.set_xlabel('Community')
ax.set_ylabel('Compile Time (ms)')
ax.set_title('Compile Time per Community')
ax.legend()

# 3. Cohesion vs Coupling scatter
ax = axes[1, 0]
sc = ax.scatter(
    comm_df['coupling_ratio'], comm_df['cohesion_ratio'],
    s=comm_df['target_count'] * 5 + 30,
    c=comm_df['compile_time_sum_ms'], cmap='YlOrRd', alpha=0.8,
)
for _, row in comm_df.iterrows():
    ax.annotate(f"C{int(row['community_id'])}",
                (row['coupling_ratio'], row['cohesion_ratio']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
plt.colorbar(sc, ax=ax, label='Compile Time (ms)')
ax.set_xlabel('Coupling Ratio (external / total edges)')
ax.set_ylabel('Cohesion Ratio (internal / total edges)')
ax.set_title('Cohesion vs Coupling (size = target count)')

# 4. SLOC per community with codegen target count overlay
ax = axes[1, 1]
sns.barplot(data=comm_df, x='community_id', y='code_lines_total', color='steelblue', ax=ax)
ax2 = ax.twinx()
ax2.plot(range(len(comm_df)), comm_df['codegen_target_count'].values,
         color='darkorange', marker='o', label='Codegen target count')
ax2.set_ylabel('Codegen Target Count', color='darkorange')
ax.set_xlabel('Community')
ax.set_ylabel('Total SLOC')
ax.set_title('SLOC per Community (line = codegen target count)')

plt.tight_layout()
plt.show()

print(f"\nTotal communities: {n_communities}")
print(f"Mean targets per community: {comm_df['target_count'].mean():.1f}")
print(f"Mean cohesion: {comm_df['cohesion_ratio'].mean():.3f}")
print(f"Mean coupling: {comm_df['coupling_ratio'].mean():.3f}")

## Bridging Targets

Betweenness centrality — high-centrality targets are split candidates.


In [ ]:
from matplotlib.patches import Patch

# Recompute betweenness centrality on G_und (not reuse directed-graph values)
bc_und = nx.betweenness_centrality(G_und, normalized=True, seed=42)

# Flag targets that bridge different communities
bridging_rows = []
for node, bc in bc_und.items():
    comm_of_node = node_comm.get(node, -1)
    neighbour_comms = {node_comm.get(v, -1) for v in G_und.neighbors(node)} - {comm_of_node}
    bridging_rows.append({
        'cmake_target': node,
        'community_id': comm_of_node,
        'betweenness_centrality': bc,
        'neighbour_communities': len(neighbour_comms),
    })

bc_df = pd.DataFrame(bridging_rows).sort_values('betweenness_centrality', ascending=False)
bc_df = bc_df.merge(
    target_df[['cmake_target', 'compile_time_sum_ms', 'code_lines_total',
               'codegen_file_count', 'fan_in', 'fan_out']],
    on='cmake_target', how='left',
)
bc_df['is_codegen'] = bc_df['codegen_file_count'].fillna(0) > 0

bridging_candidates = bc_df[bc_df['neighbour_communities'] >= 1].copy()
top_bridging = bridging_candidates.head(20)

print(f"Targets connecting >= 1 other community: {len(bridging_candidates)}")
print("\nTop bridging targets by betweenness centrality:")
display(top_bridging[['cmake_target', 'community_id', 'betweenness_centrality',
                       'neighbour_communities', 'compile_time_sum_ms',
                       'is_codegen']].reset_index(drop=True))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Distribution
ax1.hist(bc_df['betweenness_centrality'], bins=50, color='steelblue', alpha=0.7)
threshold = bc_df['betweenness_centrality'].quantile(0.95)
ax1.axvline(threshold, color='red', linestyle='--', label=f'95th pctl ({threshold:.4f})')
ax1.set_xlabel('Betweenness Centrality')
ax1.set_ylabel('Count')
ax1.set_title('Betweenness Centrality Distribution (G_und, direct edges)')
ax1.legend()

# Top-N bar chart
top_n = bc_df.head(20).copy()
bar_colors = top_n['is_codegen'].map({True: 'darkorange', False: 'steelblue'})
ax2.barh(range(len(top_n)), top_n['betweenness_centrality'], color=bar_colors)
ax2.set_yticks(range(len(top_n)))
ax2.set_yticklabels(top_n['cmake_target'], fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Betweenness Centrality')
ax2.set_title('Top 20 Targets by Betweenness Centrality (orange = codegen)')
ax2.legend(handles=[
    Patch(color='steelblue', label='Authored'),
    Patch(color='darkorange', label='Codegen'),
], loc='lower right')

plt.tight_layout()
plt.show()

high_bc = bc_df[bc_df['betweenness_centrality'] > threshold]
print(f"\nHigh-centrality split candidates (top 5%): {len(high_bc)}")
codegen_bridges = high_bc[high_bc['is_codegen']]
print(f"  of which are codegen targets: {len(codegen_bridges)}")

## Codegen Community Patterns

Do codegen targets cluster within specific communities?


In [ ]:
from scipy.stats import chi2_contingency

# Codegen target distribution across communities
comm_codegen = []
for cid, comm in enumerate(communities):
    n_total = len(comm)
    n_codegen = len(comm & codegen_targets_set)
    codegen_pct = n_codegen / n_total * 100 if n_total > 0 else 0.0
    comm_codegen.append({
        'community_id': cid,
        'target_count': n_total,
        'codegen_count': n_codegen,
        'non_codegen_count': n_total - n_codegen,
        'codegen_pct': round(codegen_pct, 1),
    })

codegen_comm_df = pd.DataFrame(comm_codegen).sort_values('codegen_pct', ascending=False)
overall_codegen_pct = len(codegen_targets_set) / G.number_of_nodes() * 100

print(f"Overall codegen target rate: {overall_codegen_pct:.1f}%")
print("\nCodegen distribution per community:")
display(codegen_comm_df)

# Chi-squared test for non-uniform distribution
contingency = codegen_comm_df[['codegen_count', 'non_codegen_count']].values
if contingency.sum() > 0 and contingency.shape[0] >= 2:
    chi2, p_value, dof, _ = chi2_contingency(contingency)
    print("\nChi-squared test for non-uniform codegen distribution:")
    print(f"  chi2={chi2:.2f}, p={p_value:.4f}, dof={dof}")
    if p_value < 0.05:
        print("  => Codegen targets are NON-uniformly distributed (clustered)")
    else:
        print("  => Codegen targets appear uniformly distributed")
else:
    print("\nInsufficient data for chi-squared test.")

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Stacked bar: authored vs codegen per community
x = codegen_comm_df['community_id'].astype(str)
ax1.bar(x, codegen_comm_df['non_codegen_count'], color='steelblue', label='Authored targets')
ax1.bar(x, codegen_comm_df['codegen_count'], color='darkorange',
        bottom=codegen_comm_df['non_codegen_count'], label='Codegen targets', alpha=0.9)
ax1.set_xlabel('Community')
ax1.set_ylabel('Target Count')
ax1.set_title('Target Distribution: Authored vs Codegen per Community')
ax1.legend()

# Codegen % per community with overall reference line
ax2.bar(codegen_comm_df['community_id'].astype(str),
        codegen_comm_df['codegen_pct'], color='darkorange', alpha=0.8)
ax2.axhline(overall_codegen_pct, color='red', linestyle='--',
            label=f'Overall avg ({overall_codegen_pct:.1f}%)')
ax2.set_xlabel('Community')
ax2.set_ylabel('Codegen Targets (%)')
ax2.set_title('Codegen Concentration by Community')
ax2.legend()

plt.tight_layout()
plt.show()

# Identify codegen-heavy communities
mean_pct = codegen_comm_df['codegen_pct'].mean()
std_pct = codegen_comm_df['codegen_pct'].std()
threshold_pct = mean_pct + std_pct
codegen_heavy = codegen_comm_df[codegen_comm_df['codegen_pct'] > threshold_pct]
if not codegen_heavy.empty:
    print(f"\nCodegen-heavy communities (>{threshold_pct:.1f}% codegen):")
    display(codegen_heavy)
else:
    print(f"\nNo communities are disproportionately codegen-heavy (threshold: {threshold_pct:.1f}%).")

## Visualisation

DAG coloured by community, codegen targets marked.


In [ ]:
import matplotlib.cm as cm
from matplotlib.patches import Patch

# ── Colour palette ───────────────────────────────────────────────────────────
cmap = cm.get_cmap('tab20') if n_communities <= 20 else cm.get_cmap('hsv')
community_colors = {cid: cmap(cid / max(n_communities - 1, 1)) for cid in range(n_communities)}
node_colors = [community_colors[node_comm.get(n, 0)] for n in G_und.nodes()]
node_sizes = [80 if n in codegen_targets_set else 20 for n in G_und.nodes()]
node_edge_colors = ['darkorange' if n in codegen_targets_set else 'none' for n in G_und.nodes()]

# ── Layout ───────────────────────────────────────────────────────────────────
n_nodes = G_und.number_of_nodes()
k_val = 2.0 / np.sqrt(max(n_nodes, 1))
pos = nx.spring_layout(G_und, k=k_val, iterations=30, seed=42)

# ── Figure 1: Full graph coloured by community ──────────────────────────────
fig, ax = plt.subplots(figsize=(18, 14))
nx.draw_networkx(
    G_und, pos=pos, ax=ax,
    node_color=node_colors, node_size=node_sizes,
    edgecolors=node_edge_colors, linewidths=1.0,
    edge_color='grey', alpha=0.6,
    with_labels=n_nodes <= 50,
    font_size=7, width=0.3,
)
legend_patches = [
    Patch(facecolor=community_colors[cid],
          label=f'C{cid} ({len(communities[cid])} targets)')
    for cid in range(n_communities)
]
legend_patches.append(
    Patch(facecolor='white', edgecolor='darkorange', linewidth=2,
          label='Codegen target (larger node)')
)
ax.legend(handles=legend_patches, loc='upper left', fontsize=8,
          ncol=max(1, n_communities // 10))
ax.set_title('Dependency Graph Coloured by Community\n'
             '(direct edges only; orange outline = codegen target)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

# ── Figure 2: Community-level summary graph ──────────────────────────────────
C = nx.Graph()
for cid in range(n_communities):
    comm = communities[cid]
    compile_ms = sum(
        int(target_metrics.loc[n, 'compile_time_sum_ms'])
        for n in comm if n in target_metrics.index
        and pd.notna(target_metrics.loc[n, 'compile_time_sum_ms'])
    )
    codegen_n = sum(1 for n in comm if n in codegen_targets_set)
    C.add_node(cid, size=len(comm), compile_ms=compile_ms, codegen_count=codegen_n)

for u, v in G_und.edges():
    cu, cv = node_comm.get(u, -1), node_comm.get(v, -1)
    if cu != cv and cu >= 0 and cv >= 0:
        if C.has_edge(cu, cv):
            C[cu][cv]['weight'] += 1
        else:
            C.add_edge(cu, cv, weight=1)

c_pos = nx.kamada_kawai_layout(C)
c_sizes = [C.nodes[cid]['size'] * 15 + 100 for cid in C.nodes()]
c_colors = [community_colors[cid] for cid in C.nodes()]
edge_weights = [C[u][v]['weight'] for u, v in C.edges()]
max_w = max(edge_weights) if edge_weights else 1

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(C, c_pos, ax=ax, node_size=c_sizes, node_color=c_colors, alpha=0.9)
if edge_weights:
    nx.draw_networkx_edges(C, c_pos, ax=ax,
                           width=[2 + 6 * w / max_w for w in edge_weights],
                           edge_color='grey', alpha=0.6)
nx.draw_networkx_labels(
    C, c_pos, ax=ax,
    labels={cid: f"C{cid}\n({C.nodes[cid]['size']})" for cid in C.nodes()},
    font_size=9, font_weight='bold',
)
ax.set_title('Community-Level Summary Graph\n'
             '(node size = target count, edge width = cross-community edges)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

for cid in sorted(C.nodes()):
    n = C.nodes[cid]
    ext = sum(C[cid][nb]['weight'] for nb in C.neighbors(cid))
    print(f"C{cid}: {n['size']} targets, {n['codegen_count']} codegen, "
          f"compile={n['compile_ms']:,}ms, cross-community edges={ext}")

## Merge Candidates

Communities with high internal coupling and low external coupling.


In [ ]:
from build_optimiser.simulation import simulate_merge

# ── Identify merge candidates: high cohesion, low coupling ──────────────────
COHESION_THRESHOLD = comm_df['cohesion_ratio'].quantile(0.50)
COUPLING_THRESHOLD = comm_df['coupling_ratio'].quantile(0.50)
MIN_TARGETS = 2

candidates = comm_df[
    (comm_df['cohesion_ratio'] >= COHESION_THRESHOLD)
    & (comm_df['coupling_ratio'] <= COUPLING_THRESHOLD)
    & (comm_df['target_count'] >= MIN_TARGETS)
].copy()

print(f"Merge candidate communities (cohesion >= {COHESION_THRESHOLD:.3f}, "
      f"coupling <= {COUPLING_THRESHOLD:.3f}, targets >= {MIN_TARGETS}):")
display(candidates[['community_id', 'target_count', 'cohesion_ratio',
                     'coupling_ratio', 'compile_time_sum_ms',
                     'codegen_target_count']].reset_index(drop=True))

# ── Run simulate_merge for each candidate community ──────────────────────────
sim_results = []
for _, row in candidates.iterrows():
    cid = int(row['community_id'])
    comm = communities[cid]
    targets = [t for t in comm if t in target_metrics.index]
    if len(targets) < 2:
        continue

    result = simulate_merge(G, targets, target_df)
    sim_results.append({
        'community_id': cid,
        'targets': ', '.join(sorted(targets)),
        'target_count': len(targets),
        'cohesion_ratio': row['cohesion_ratio'],
        'coupling_ratio': row['coupling_ratio'],
        'before_ms': result['before_ms'],
        'after_ms': result['after_ms'],
        'savings_ms': result['savings_ms'],
        'savings_pct': result['savings_ms'] / result['before_ms'] * 100
                       if result['before_ms'] > 0 else 0.0,
        'notes': '; '.join(result['notes']),
    })

sim_df = pd.DataFrame(sim_results).sort_values('savings_ms', ascending=False)
display(sim_df)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Cohesion/coupling quadrant with merge candidates highlighted
ax1.scatter(
    comm_df['coupling_ratio'], comm_df['cohesion_ratio'],
    s=comm_df['target_count'] * 8 + 30,
    color='steelblue', alpha=0.6, label='All communities',
)
if not candidates.empty:
    ax1.scatter(
        candidates['coupling_ratio'], candidates['cohesion_ratio'],
        s=candidates['target_count'] * 8 + 30,
        color='darkorange', alpha=0.9, label='Merge candidates', zorder=5,
    )
    for _, row in candidates.iterrows():
        ax1.annotate(f"C{int(row['community_id'])}",
                     (row['coupling_ratio'], row['cohesion_ratio']),
                     textcoords='offset points', xytext=(4, 4), fontsize=8)
ax1.axvline(COUPLING_THRESHOLD, color='red', linestyle='--', alpha=0.6)
ax1.axhline(COHESION_THRESHOLD, color='red', linestyle='--', alpha=0.6)
ax1.set_xlabel('Coupling Ratio')
ax1.set_ylabel('Cohesion Ratio')
ax1.set_title('Merge Candidates: Cohesion vs Coupling')
ax1.legend()

# Expected savings from merging
if not sim_df.empty:
    x_labels = [f"C{int(r)}" for r in sim_df['community_id']]
    ax2.bar(x_labels, sim_df['savings_ms'], color='darkorange', alpha=0.8)
    ax2.set_xlabel('Community')
    ax2.set_ylabel('Estimated Savings (ms)')
    ax2.set_title('Estimated Build Time Savings from Merge')
    for i, (_, row) in enumerate(sim_df.iterrows()):
        ax2.text(i, row['savings_ms'] + 10, f"{row['savings_pct']:.1f}%",
                 ha='center', fontsize=8)
else:
    ax2.text(0.5, 0.5, 'No merge candidates', transform=ax2.transAxes, ha='center')
    ax2.set_title('Estimated Build Time Savings from Merge')

plt.tight_layout()
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────────
if not sim_df.empty:
    total_savings = sim_df['savings_ms'].sum()
    total_build = target_df['total_build_time_ms'].sum()
    print(f"\nTotal estimated savings from all merges: {total_savings:,} ms")
    if total_build > 0:
        print(f"  ({total_savings / total_build * 100:.2f}% of total build time)")
    print("\nTop candidates:")
    for _, row in sim_df.head(5).iterrows():
        print(f"  C{int(row['community_id'])}: merge {int(row['target_count'])} targets "
              f"({row['targets']}) -> save {int(row['savings_ms']):,} ms ({row['savings_pct']:.1f}%)")
        if row['notes']:
            for note in row['notes'].split('; '):
                print(f"    note: {note}")
else:
    print("\nNo communities meet merge criteria at current thresholds.")
    print(f"Consider lowering cohesion threshold (currently {COHESION_THRESHOLD:.3f}) "
          f"or raising coupling threshold (currently {COUPLING_THRESHOLD:.3f}).")